# Next-Word Predictor using LSTM (TensorFlow / Keras)

## Problem
Given a partial sentence, predict the most likely **next word** and, by feeding each prediction back in, generate the next *N* words. This is the core idea behind keyboard autocomplete and text-suggestion systems.

## Input → Output
| | Description | Example |
|---|---|---|
| **Input** | Seed text (any length) + number of words to generate | `"You are motivated"`, `num_words=8` |
| **Output** | Seed text extended word by word with the model's predictions | `"You are motivated and independent strive to contribute with new ideas"` |

## Datasets
1. **Kaggle – Next Word Prediction** (`ronikdedhia/next-word-prediction`, file `1661-0.txt`): a large book-length text corpus.
2. **CMS / HEP job vacancy text** (`cms_vac_data.txt`): a small corpus (30 lines, 242 unique words) taken from a high-energy-physics job site.

## Pipeline
1. **Data loading**: download with `kagglehub`, read the file, drop empty lines.
2. **Tokenization**: Keras `TextVectorization` (lowercase, strip punctuation, word → integer ID); probe the corpus first to choose a sensible `VOCAB_SIZE`.
3. **N-gram sequence generation**: a sliding window turns each line into growing prefixes (`w1 w2`, `w1 w2 w3`, …). The small corpus gives 435 training sequences.
4. **Padding**: `pad_sequences(..., padding='pre')` so the target word is always in the last column.
5. **X / y split**: X = all tokens except the last, y = the last token (the next word).
6. **Model**: `Embedding → LSTM → Dropout(0.3) → Dense(vocab_size, softmax)`.
7. **Training**: `sparse_categorical_crossentropy` loss, Adam optimizer, 50 epochs, batch size 24.
8. **Inference**: autoregressive greedy decoding (`argmax`), where each predicted word is appended to the seed and fed back in.

## Results
- Training accuracy reaches **~97%** after 50 epochs on the small corpus (loss ≈ 0.58).
- With 10 epochs the predictions were poor; from about 20 epochs on they became coherent.
- Stacked (deep) LSTM layers did **not** improve results on the small data.
- The model **overfits**: it memorizes the training text (high variance), which is expected for a corpus this small.

## Tech Stack
Python · TensorFlow / Keras · LSTM · NumPy · kagglehub · Jupyter

## Modular Version
A refactored, script-based version of this project is in [`LSTM-next-word-predictor/`](LSTM-next-word-predictor/). It has `config`, `data`, `preprocessing`, `model`, `train`, `evaluation` (perplexity and loss/accuracy curves) and `predict` modules, and saves the model and vocabulary as artifacts.


In [ ]:
from  pathlib import Path
import os 
lstmDataPath = r"/home/mrafiku/AI_learning/deep_learning/data/lstm_text_data"
datapath = Path(lstmDataPath)
print(datapath)

In [ ]:
# !pip install kagglehub

In [ ]:
# import kagglehub
# path = kagglehub.dataset_download("ronikdedhia/next-word-prediction")
# print("Path to dataset files:", path)

import kagglehub

path = kagglehub.dataset_download(
    "ronikdedhia/next-word-prediction",
    # output_dir="./AI_learning/deep_learning/data/lstm_text_data"   # your desired path
    output_dir= datapath # here my path where I want to download the dataset.
)
print("Downloaded to:", path)

In [ ]:
import os

print(os.listdir(datapath))   # 'path' is what kagglehub.dataset_download() returned

In [ ]:
# with open(os.path.join(datapath, "1661-0.txt"), "r", encoding="utf-8") as f:
with open(os.path.join(datapath, "cms_vac_data.txt"), "r", encoding="utf-8") as f:
    raw_text = f.read()

text_data = raw_text.split("\n")   # one string per line
text_data = [line for line in text_data if line.strip() != ""]  # drop empty lines
print(f"type of the data format:{type(text_data)}")

In [ ]:
print(f"Number of lines in the text data: {len(text_data)}")
print(f"First three lines of the text data: {text_data[1:4]}" )

In [ ]:

max_length_sentences = max([len(sentences) for sentences in text_data ])
print(f"max length sentence avialable in the text data: {max_length_sentences}")

### Here We downloaded the data from kaggle and imported and extracted data in text list. 
- known as text_data

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization, Dense

## This is just to decide vocab size for use further.
- Run this below cell first.
- it tells you the real vocabulary size of this dataset, so your next choice isn't arbitrary.

### When you call ``.adapt()``, you are forcing the layer to read through your entire text_data dataset.

During this reading phase, the layer automatically does a lot of heavy lifting behind the scenes:

- It ``converts`` all the text to ``lowercase``.

- It ``strips away`` all ``punctuation``.
    - By default, when Keras's TextVectorization layer runs its automatic punctuation stripping, - it removes the standard set of 32 ASCII punctuation characters.
    - Here is the exact list of the ``punctuation marks`` it strips away:
``! " # $ % & ' ( ) * + , - . / : ; < = > ? @ [ \ ] ^ _  { | } ~` ``

- It ``splits`` the sentences into ``individual words``.

- It counts how many times every single word appears and sorts them by popularity to build its internal dictionary.

In our example:
When it reads ["hello world", "hello friend"], it sees that "hello" appears twice,

while "world" and "friend" appear once. It officially memorizes these words.

In [ ]:
from tensorflow.keras.layers import TextVectorization

# First, check how many unique words actually exist, so vocab_size isn't a guess
probe_layer = TextVectorization(output_mode='int')   # no max_tokens cap, for counting only
probe_layer.adapt(text_data) # .adapt()
print("Unique words in corpus:", len(probe_layer.get_vocabulary()))

``print("Unique words in corpus:", len(probe_layer.get_vocabulary()))``
``What it does:``

- This line is a diagnostic tool to tell you exactly how many unique words the layer just found and memorized during the .adapt() phase.

- probe_layer.get_vocabulary() spits out the actual Python list of the sorted words it memorized.

- len(...) is a standard Python function that just counts how many items are inside that list.

**In our example**

- If we run probe_layer.get_vocabulary(), it outputs this list:
['', '[UNK]', 'hello', 'friend', 'world']

- (Remember: Keras always secretly adds an empty space '' at index 0 for padding, and an Unknown token '[UNK]' at index 1 for words it hasn't seen).

- When you wrap that list in the len() function, Python counts the items (1, 2, 3, 4, 5) and prints the final number.

### **The Lean Model (vocab_size = 5000 to 8000)**
- If you want your model to train faster and be less prone to overfitting, you can aggressively cut the vocabulary down to 5,000.

- Why do this? Remember the rule of thumb! Even if you cut your vocabulary in half, the top 5,000 most popular words will still cover well over 95% of the total words on the page.

- This forces the neural network to ignore rare, distracting words and focus entirely on the strong, common patterns in the text. (For standard sentiment analysis, 5,000 is often more than enough to get excellent accuracy.


In [ ]:
# VOCAB_SIZE = 8000
VOCAB_SIZE = 200 # for new data
# maxlen = 20 
vectorize_layer = TextVectorization(
    max_tokens =VOCAB_SIZE,
    output_mode = 'int',
    # output_sequence_length = maxlen
)
vectorize_layer.adapt(text_data)  # # scans text_data, builds the word→ID mapping


In [ ]:
# vocab = vectorize_layer.get_vocabulary()
# print(len(vocab))
# print(f"word -token:{vocab[:10]}")
# print(f"vocab index at place 0: {vocab[0]}")
# print(vocab)
idcount = 0
for line in text_data:
    idcount += 1
    print(f"line {idcount}: {line}")
    if idcount == 5:
        break   
    # print(f"vectorized line: {vectorize_layer(lin)}")

## Step 2 — Build n-gram sequences

Imagine our text_data contains just one single sentence:

["cats are very loud"]

And imagine your vectorize_layer has already learned these integer IDs:

cats = 14, are = 8, very = 42, loud = 55

Step-by-Step Breakdown
1. input_sequences = []
What it does: Creates an empty list. This will act as the master container where we store all of our newly generated sequences.

2. for line in text_data:
What it does: Starts a loop to process your dataset one sentence (line) at a time.

In our example: It grabs the string "cats are very loud".
```python
token_list = vectorize_layer([line]).numpy()[0]

    - What it does: This pushes the raw text sentence through your vectorizer to convert the words into their integer IDs.

    - Because the layer outputs a 2D TensorFlow Tensor (e.g., [[14, 8, 42, 55, 0, 0]]), .numpy() converts it to a standard array, and [0] extracts the first (and only) flat list from inside that 2D structure.

In our example: token_list becomes [14, 8, 42, 55, 0, 0]. (Note: The zeros at the end exist if your layer was configured with a fixed output sequence length to pad shorter sentences).

In [ ]:
input_sequences = []

for line in text_data:
    token_list = vectorize_layer([line]).numpy()[0]   # tokenize this one line
    token_list = token_list[token_list != 0]           # strip any stray zero/padding tokens

    for i in range(1, len(token_list)):  # The inner for i in range(1, len(token_list)) loop is the actual "sliding window" — 
        # for a 6-word line, it produces 5 growing sub-sequences.
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

print(f"Total training sequences: {len(input_sequences)}")

### Step 3 — Pad sequences

- maxlen=max_sequence_len — pads/truncates every sequence to match your longest n-gram. 
- If your scripture text has some very long lines, this can end up large — worth printing max_sequence_len to sanity-check
- it's not absurdly big (if it is, consider capping it manually with a smaller maxlen, accepting some truncation on outlier lines).
- ``padding='pre'`` — zeros go at the front. This is required here (not optional) because.
-  Step 4 always grabs the last column as y; if you padded 'post' instead, short sequences would have their real last word buried in the middle, not at the end.

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

max_sequence_len = max(len(seq) for seq in input_sequences)
print(f"max sequence length: {max_sequence_len}")

input_sequences = np.array(
    pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre')
)

### Step 4 — Split X and y

In [ ]:
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

print("X shape:", X.shape)   # (num_sequences, max_sequence_len - 1)
print("y shape:", y.shape)   # (num_sequences,)

### Step 5 — Build the LSTM model
Embedding(input_dim=vocab_size, ...) — must match the vocab_size you set in Step 1; mismatches here are a common source of index out of range errors later.

input_length=max_sequence_len - 1 — matches X's shape exactly (X has one fewer column than the full padded sequence, since y took the last one).

LSTM(150) — 150 units is a reasonable starting point for a moderately sized corpus; more units = more capacity but slower training and more overfitting risk on a small dataset.

Dense(vocab_size, activation='softmax') — one output neuron per word in your vocabulary. This is the layer size that changes most dramatically compared to your sentiment model's Dense(1, sigmoid).

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dropout, Embedding, LSTM

model = Sequential([
    Input(shape=(max_sequence_len - 1,)),
    # Embedding(input_dim=VOCAB_SIZE, output_dim=100),
    Embedding(input_dim=VOCAB_SIZE, output_dim=60),
    # LSTM(150, return_sequences=True),
    # LSTM(150), # this is for the large dataset.
    LSTM(50), # for tiny datasets.
    Dropout(0.3), # --> "Randomly turn off 30% of the connections coming out of the LSTM layer."
    # specific Dropout layer directly after your LSTM finishes thinking, but before it makes its final prediction.
    Dense(VOCAB_SIZE, activation='softmax') 
    # because we want to predict the next word, which is a classification problem over the vocabulary
])


In [ ]:
model.summary()

### Step 6 — Compile

In [ ]:
model.compile(
    loss='sparse_categorical_crossentropy',   # y is a raw integer (word ID), not one-hot
    optimizer='adam',
    metrics=['accuracy']
)

### Step 7 — Train
batch_size=64 — with a scripture-length text, you likely have far more training sequences than your FAQ corpus did,

so training will take noticeably longer per epoch; lower batch_size (e.g. 32) if you run into memory issues.

Watch the printed accuracy each epoch — if it plateaus low (well under 50%), 

the sequences may be too long/complex for LSTM(150); if it climbs toward 90%+ quickly, the model is memorizing this specific text (expected and fine for this exercise).

In [ ]:
# history = model.fit(X, y, epochs=20, batch_size=64, verbose=1) #EPOCHS = 50
history = model.fit(X, y, epochs=50, batch_size=24, verbose=1) #EPOCHS = 50

In [ ]:
# CHECKING THE ACTUAL TEXT.
icount = 5
for line in text_data[icount:-1]:
    print(f"line {icount}: {line}")
    icount += 1
   

### Step 8 — Generate next words

In [ ]:
def predict_next_words(seed_text, num_words=5):
    for _ in range(num_words):
        token_list = vectorize_layer([seed_text]).numpy()[0]
        token_list = token_list[token_list != 0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len - 1, padding='pre')

        predicted_probs = model.predict(token_list, verbose=0)[0]
        predicted_index = np.argmax(predicted_probs)
        predicted_word = vectorize_layer.get_vocabulary()[predicted_index]

        seed_text += " " + predicted_word
    return seed_text


In [ ]:
text_input = "You are motivated"
print(predict_next_words(text_input, num_words=8))
# print(predict_next_words(" My own complete happiness", num_words=15))


## Conclusion:
- while i used epochs =10 it was not predicting correctly.
- I increased epochs to 20, it's start predicting correctly with the same input as i tried earlier with 10 epochs.
- I also tested this on the other cms vacancy data text taken from the hep website.
- while using the stacked/deep lstm not improving performance even versoning.
- overfitting/high variance problem, it memorize the data. 